# Chat with MiniCPM5-2B and OpenVINO

[MiniCPM5-2B](https://huggingface.co/openbmb/MiniCPM5-2B) is a dense 2B-parameter language model from OpenBMB's [MiniCPM5](https://github.com/OpenBMB/MiniCPM) series, designed for on-device deployment, local assistants, and resource-constrained scenarios. Despite its compact footprint, it reaches 2B-class open-source SOTA (average score **53.9** across nine benchmark groups, ahead of larger 4B-class models in its comparison set) and remains competitive with 4B-class models overall.

**Key Features of MiniCPM5-2B:**

- **Hybrid Reasoning** — The model produces a `<think>...</think>` reasoning trace before its final answer by default. Thinking can be disabled per-request through the `enable_thinking` chat-template flag for fast, direct answers.
- **2B-Class SOTA Quality** — Post-trained with SFT, RL and On-Policy Distillation (OPD) over the open [UltraData](https://ultradata.openbmb.cn/) corpus; strongest in coding, math, long-context understanding, tool use and agentic tasks within its size class.
- **Long Context** — Native support for up to **131,072 tokens** with RoPE theta of 5M.
- **Efficient Architecture** — A standard `LlamaForCausalLM` backbone with 42 layers, 2048 hidden size and Grouped Query Attention (16 query heads, 2 KV heads) for memory-efficient inference.
- **Tool Calling** — A native function-calling chat template for agentic workflows.

More details can be found in the [model card](https://huggingface.co/openbmb/MiniCPM5-2B), the [MiniCPM tech report](https://arxiv.org/pdf/2506.07900) and the [GitHub repository](https://github.com/OpenBMB/MiniCPM).

In this tutorial, we consider how to convert and compress MiniCPM5-2B using [Optimum Intel](https://github.com/huggingface/optimum-intel) with [NNCF](https://github.com/openvinotoolkit/nncf) weight compression, run text generation with the [OpenVINO GenAI](https://github.com/openvinotoolkit/openvino.genai) `LLMPipeline` on CPU and GPU, toggle the model's thinking mode, and launch an interactive Gradio chat.



## Installation Instructions

This is a self-contained example that relies solely on its own code.</br>
We recommend running the notebook in a virtual environment. You only need a Jupyter server to start.
For further details, please refer to [Installation Guide](../../README.md).

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select model](#Select-model)
- [Convert and Optimize model](#Convert-and-Optimize-model)
- [Select inference device](#Select-inference-device)
- [Create the inference pipeline](#Create-the-inference-pipeline)
- [Chat with the model](#Chat-with-the-model)
    - [Fast mode: disable thinking](#Fast-mode:-disable-thinking)
- [Interactive demo](#Interactive-demo)



<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/minicpm5/minicpm5.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

Install required packages and fetch helper utilities.


In [1]:
%pip install -q "optimum-intel[openvino,nncf] @ git+https://github.com/huggingface/optimum-intel.git" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "openvino-genai" "transformers>=5.6" "gradio>=6.0,<7" "ipywidgets" "datasets"


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Note: you may need to restart the kernel to use updated packages.


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import requests

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)


## Select model
[back to top ⬆️](#Table-of-contents:)

We use the final [openbmb/MiniCPM5-2B](https://huggingface.co/openbmb/MiniCPM5-2B) checkpoint (BF16, post-trained with RL + OPD). OpenBMB also publishes an [SFT-only checkpoint](https://huggingface.co/openbmb/MiniCPM5-2B-SFT) and a family of smaller/larger variants, but this tutorial focuses on the final 2B release.

The table below summarizes the available weight formats for the exported OpenVINO model:

| Format | Description |
|---|---|
| **FP16** | Half-precision weights, best quality, largest footprint (~5 GB) |
| **INT8** | 8-bit weight compression, near-FP16 quality, ~2.6 GB |
| **INT4** | 4-bit weight compression with NNCF, best size/speed trade-off (~1.7 GB) |


In [3]:
import ipywidgets as widgets

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("minicpm5.ipynb")

weight_format = widgets.Dropdown(
    options=["FP16", "INT8", "INT4"],
    value="INT4",
    description="Weight format:",
    disabled=False,
)

weight_format


Dropdown(description='Weight format:', index=2, options=('FP16', 'INT8', 'INT4'), value='INT4')

In [4]:
print(f"Selected weight format: {weight_format.value}")
pt_model_id = "openbmb/MiniCPM5-2B"
model_dir = Path(pt_model_id.split("/")[-1])
model_export_dir = model_dir / weight_format.value
print(f"Export directory: {model_export_dir}")


Selected weight format: INT4
Export directory: MiniCPM5-2B/INT4


## Convert and Optimize model
[back to top ⬆️](#Table-of-contents:)

MiniCPM5-2B is a PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). For convenience, we will use OpenVINO integration with HuggingFace Optimum.

🤗 [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) is the interface between the 🤗 Transformers and Diffusers libraries and the different tools and libraries provided by Intel to accelerate end-to-end pipelines on Intel architectures.

`optimum-cli` provides a command line interface for model conversion and optimization. NNCF weight compression will be applied on the fly when a compressed `weight-format` is selected. INT4 compression reduces the model footprint roughly 3x compared to FP16 while keeping generation quality close to the original — a good default for on-device scenarios MiniCPM5 targets.


In [5]:
from cmd_helper import optimum_cli

if not (model_export_dir / "openvino_model.xml").exists():
    additional_args = {}
    if weight_format.value != "FP16":
        additional_args["weight-format"] = weight_format.value.lower()
    optimum_cli(pt_model_id, model_export_dir, additional_args=additional_args)
else:
    print(f"Model already exported to {model_export_dir}")


**Export command:**

`optimum-cli export openvino --model openbmb/MiniCPM5-2B MiniCPM5-2B/INT4 --weight-format int4`

In [6]:
MODEL_TOTAL_PARAMETERS = 2_516_756_480
fp16_footprint = MODEL_TOTAL_PARAMETERS * 2 / 2**30

exported_size = (model_export_dir / "openvino_model.bin").stat().st_size / 2**30
print(f"Exported model size ({weight_format.value}): {exported_size:.2f} GB")
print(f"FP16 reference footprint: {fp16_footprint:.2f} GB")
print(f"Memory footprint reduction: {fp16_footprint / exported_size:.1f}x")


Exported model size (INT4): 1.49 GB
FP16 reference footprint: 4.69 GB
Memory footprint reduction: 3.1x


## Select inference device
[back to top ⬆️](#Table-of-contents:)

We use [LLMPipeline](https://docs.openvino.ai/2025/documentation/openvino-genai/llm-inference.html) from OpenVINO GenAI for inference. It loads the exported OpenVINO IR model and provides a simple `generate()` API with built-in tokenization and chat-template support. MiniCPM5-2B runs on both CPU and Intel GPU with this pipeline; the NPU device is not supported by this example.


In [7]:
from notebook_utils import device_widget

device = device_widget(default="AUTO", exclude=["NPU"])
device


Dropdown(description='Device:', index=3, options=('CPU', 'GPU.0', 'GPU.1', 'AUTO'), value='AUTO')

## Create the inference pipeline
[back to top ⬆️](#Table-of-contents:)

`LLMPipeline` is the main object for text generation with OpenVINO GenAI. It can be constructed straight from the directory with the converted model. We also define a small helper that streams generated tokens to the notebook output as they are produced.


In [8]:
import openvino_genai as ov_genai

print(f"Loading model from {model_export_dir} on {device.value} ...")
pipe = ov_genai.LLMPipeline(str(model_export_dir), device=device.value)


def make_streamer(tokenizer: ov_genai.Tokenizer) -> ov_genai.TextStreamer:
    def print_chunk(text: str) -> ov_genai.StreamingStatus:
        print(text, end="", flush=True)
        return ov_genai.StreamingStatus.RUNNING

    return ov_genai.TextStreamer(tokenizer, print_chunk)


Loading model from MiniCPM5-2B/INT4 on AUTO ...


## Chat with the model
[back to top ⬆️](#Table-of-contents:)

MiniCPM5-2B is a hybrid reasoning model: by default it first generates a `<think>...</think>` reasoning trace and only then produces the final answer. The chat template applies the recommended sampling settings from the model card (`temperature=1.0`, `top_p=0.95`, plus `repetition_penalty=1.05` for repetitive outputs).


In [9]:
config = ov_genai.GenerationConfig(max_new_tokens=512, temperature=1.0, top_p=0.95, repetition_penalty=1.05)
pipe.generate("Give me three ideas for a weekend project, with a one-line plan for each.", config, streamer=make_streamer(pipe.get_tokenizer()))


<think>


We

 need

 to

 respond

...

 a

 simple

 notebook

.

"<think>\nWe need to respond with three ideas for a weekend project, each with a one-line plan. The user wants three ideas and a one-line plan for each. So we should provide three ideas, each with a short description or plan in one line. We can make them simple and fun. Let's think of some weekend projects: maybe a garden, a craft, a small project. We need to keep it concise.\n</think>\n\nHere are three weekend project ideas:\n\n1. **Plant a small herb garden** – Plan to buy herbs like basil and thyme and grow them in a sunny spot.\n2. **Create a mini terrarium** – Plan to collect small rocks, sand, and moss in a clear glass container.\n3. **Make a memory book** – Plan to gather photos and notes from recent trips to fill a simple notebook."

### Fast mode: disable thinking
[back to top ⬆️](#Table-of-contents:)

For simple questions the reasoning trace is unnecessary overhead. MiniCPM5-2B's chat template exposes an `enable_thinking` flag — when it is set to `False`, an empty `<think></think>` block is injected and the model answers directly. In OpenVINO GenAI the flag is passed through `extra_context` of `apply_chat_template`, and the pre-formatted prompt is then generated with `apply_chat_template=False` to avoid double formatting.


In [10]:
tokenizer = pipe.get_tokenizer()
history = [{"role": "user", "content": "Name the three most common states of matter. Answer in one line."}]
prompt = tokenizer.apply_chat_template(history, add_generation_prompt=True, extra_context={"enable_thinking": False})
fast_config = ov_genai.GenerationConfig(
    max_new_tokens=128, temperature=1.0, top_p=0.95, repetition_penalty=1.05, apply_chat_template=False
)
pipe.generate(prompt, fast_config, streamer=make_streamer(tokenizer))


Solid

,

 liquid

,

 gas

'Solid, liquid, gas'

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

Try an interactive multi-turn chat with the model. The `<think>...</think>` section is rendered in italics while streaming; the **Enable thinking** checkbox toggles the reasoning mode and the accordion exposes the main sampling parameters.


In [11]:
if not Path("gradio_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/minicpm5/gradio_helper.py")
    open("gradio_helper.py", "w").write(r.text)


In [12]:
from gradio_helper import make_demo

demo = make_demo(pipe)
try:
    demo.launch(height=600)
except Exception:
    demo.launch(share=True, height=600)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [13]:
# # Clean up resources (uncomment when finished)
# # import shutil

# # shutil.rmtree(model_dir, ignore_errors=True)
